# KV Cache (Key-Value Cache)

## 1. What is KV Cache?
KV Cache is an inference optimization technique used in autoregressive LLMs. It stores previously computed **Keys (K) and Values (V)** so they don't need to be recalculated at every generation step.

## 2. Why do we need it?
LLMs generate text one token at a time.

Without KV Cache, previous tokens' Key and Value representations are recomputed at every decoding step, causing redundant computation.

**Solution:** Store previous Keys and Values and reuse them when generating new tokens.

## 3. How does it work?

For each Transformer layer:

- Compute Query (Q), Key (K) and Value (V).
- Store K and V in the cache.
- For the next token, compute its new Q, K and V.
- Append the new K and V to the cache.
- Use the current Q with all cached K and V to calculate attention.

\[
Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

**Important:** Standard KV Cache stores Keys and Values, not Queries.

## 4. Two phases of LLM inference

**A. Prefill**
- Process the entire input prompt.
- Calculate Keys and Values for all prompt tokens.
- Populate the KV Cache.
- Produce logits for the first generated token.

**B. Decode**
- Process one new token at a time.
- Reuse previously cached Keys and Values.
- Append new Keys and Values.
- Repeat until generation finishes.

## 5. KV Cache Memory Formula

\[
\boxed{M=2LBTHd_hS}
\]

Where:

- L = Number of Transformer layers
- B = Batch size
- T = Cached sequence length
- H = Number of KV heads
- \(d_h\) = Dimension per head
- S = Bytes per element
- 2 = Key and Value tensors

**Example: DistilGPT-2**

- Layers = 6
- Batch size = 1
- Tokens = 1024
- KV heads = 12
- Head dimension = 64
- FP32 = 4 bytes

\[
M=2\times6\times1\times1024\times12\times64\times4
\]

\[
\boxed{M=36\ MiB}
\]

## 6. Attention Architectures

| Architecture | KV Heads | Main idea |
|---|---:|---|
| MHA | Equal to Query heads | Every Query head has its own K and V head |
| MQA | 1 | All Query heads share one K and V head |
| GQA | Between 1 and Query heads | Groups of Query heads share K and V heads |

**GQA** reduces KV Cache memory while retaining multiple Query heads.

## 7. PagedAttention

PagedAttention is a KV Cache memory-management technique used by vLLM.

It stores KV Cache in fixed-size blocks instead of requiring one large contiguous memory allocation per sequence.

Benefits:
- Better GPU memory utilization.
- Reduced memory fragmentation.
- Higher inference throughput.
- Efficient concurrent request handling.

## 8. Important Takeaways

1. KV Cache avoids recomputing previous tokens' Keys and Values.
2. Prefill processes the entire prompt; Decode generates tokens incrementally.
3. KV Cache improves decoding efficiency but consumes GPU memory.
4. KV Cache memory increases with sequence length and batch size.
5. GQA and MQA reduce KV Cache memory requirements.
6. PagedAttention improves KV Cache memory management.

**Remember:**

- Context Window = How much context the model can process.
- KV Cache = Previously computed Keys and Values stored for reuse.
- Prefill = Process the input prompt.
- Decode = Generate new tokens.


In [1]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

prompt = "The capital of India is"

inputs = tokenizer(prompt, return_tensors="pt")

# STEP 1: Prefill
with torch.no_grad():
    outputs = model(
        **inputs,
        use_cache=True
    )

# Retrieve KV Cache
past_key_values = outputs.past_key_values

print("Cache type:", type(past_key_values))

# Inspect each layer
for layer_idx, layer_cache in enumerate(past_key_values):

    key, value = layer_cache[:2]

    print(f"\nLayer {layer_idx}")
    print("Key shape:", key.shape)
    print("Value shape:", value.shape)

    print("Cached tokens:", key.shape[-2])

# STEP 2: Generate one new token using the cache
next_token = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True
)

with torch.no_grad():
    next_outputs = model(
        input_ids=next_token,
        past_key_values=past_key_values,
        use_cache=True
    )

print("\nNew token:", tokenizer.decode(next_token[0]))

# Check updated cache
updated_cache = next_outputs.past_key_values

print(
    "Previous cache length:",
    past_key_values.get_seq_length()
    if hasattr(past_key_values, "get_seq_length")
    else "See cache tensor shape above"
)

print(
    "Updated cache length:",
    updated_cache.get_seq_length()
    if hasattr(updated_cache, "get_seq_length")
    else updated_cache[0][0].shape[-2]
)


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Cache type: <class 'transformers.cache_utils.DynamicCache'>

Layer 0
Key shape: torch.Size([1, 12, 5, 64])
Value shape: torch.Size([1, 12, 5, 64])
Cached tokens: 5

Layer 1
Key shape: torch.Size([1, 12, 5, 64])
Value shape: torch.Size([1, 12, 5, 64])
Cached tokens: 5

Layer 2
Key shape: torch.Size([1, 12, 5, 64])
Value shape: torch.Size([1, 12, 5, 64])
Cached tokens: 5

Layer 3
Key shape: torch.Size([1, 12, 5, 64])
Value shape: torch.Size([1, 12, 5, 64])
Cached tokens: 5

Layer 4
Key shape: torch.Size([1, 12, 5, 64])
Value shape: torch.Size([1, 12, 5, 64])
Cached tokens: 5

Layer 5
Key shape: torch.Size([1, 12, 5, 64])
Value shape: torch.Size([1, 12, 5, 64])
Cached tokens: 5

New token:  the
Previous cache length: 6
Updated cache length: 6
